# Hidden Entrepreneurs Detection
## Mastercard x AIESEC Hackathon | 23 May 2026

**Задача:** Найти физических лиц, ведущих коммерческую деятельность через потребительские карты.

**Подход:**
1. Используем 25K бизнес-карт как эталон коммерческого поведения
2. Строим признаки для каждой карты из транзакций
3. Обучаем классификатор: бизнес (1) vs потребитель (0)
4. Решаем проблему **label noise** двумя методами:
   - **Итеративная очистка** — убираем подозрительных из train
   - **PU Learning (2 итерации)** — обучаем только на надёжных потребителях
5. Скорим 80K карт — кто похож на бизнес = скрытый предприниматель

**Фиксы по code review:**
- Убран data leakage: train/val/test разбивка по card_number, без пересечений
- Исправлен баг is_night: `between(22,6)` → `(hour>=22) | (hour<=6)`
- PU Learning итерируется 2 раза для устранения circular bias
- При итеративной очистке скорим только held-out consumer карты


## 1. Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, roc_auc_score,
    roc_curve, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
print('Библиотеки загружены.')


## 2. Загрузка данных

In [ ]:
biz   = pd.read_parquet('business_cards_MDQ.parquet')
cons  = pd.read_parquet('consumer_cards_MDQ.parquet')
merch = pd.read_parquet('merchants_reference.parquet')

for df in [biz, cons]:
    df['transaction_date']      = pd.to_datetime(df['transaction_date'])
    df['transaction_timestamp'] = pd.to_datetime(df['transaction_timestamp'])

biz['label']  = 1
cons['label'] = 0

print(f'Business : {biz.card_number.nunique():,} карт | {len(biz):,} транзакций')
print(f'Consumer : {cons.card_number.nunique():,} карт | {len(cons):,} транзакций')
print(f'Merchants: {len(merch):,} мерчантов')


## 3. EDA — Разведочный анализ

Находим MCC-коды которые значительно чаще встречаются у бизнеса.
`ratio` = насколько этот MCC популярнее у бизнеса чем у потребителя.


In [ ]:
biz_mcc_freq  = biz['mcc'].value_counts(normalize=True).rename('biz_%')
cons_mcc_freq = cons['mcc'].value_counts(normalize=True).rename('cons_%')
compare = pd.DataFrame([biz_mcc_freq, cons_mcc_freq]).T.fillna(0)
compare['ratio'] = compare['biz_%'] / (compare['cons_%'] + 0.0001)
compare = compare.sort_values('ratio', ascending=False)

print('Топ-15 бизнес-специфичных MCC:')
print(compare.head(15).to_string())


## 4. Feature Engineering

Сворачиваем транзакции в признаки карты (одна строка = одна карта).

**Исправлен баг:** `is_night` теперь корректно считает ночные часы (22:00-06:00).
`between(22, 6)` всегда возвращает False — исправлено на `(hour>=22) | (hour<=6)`.


In [ ]:
BUSINESS_MCC = {
    '7399','5045','5943','7379','8931',
    '7392','4812','5021','8911','4816',
    '8111','4214','5046','7372','5111',
    '7361','5968','5199','4215'
}

def build_features(df):
    """Агрегирует транзакции до уровня карты. Возвращает одну строку на карту."""
    d = df.copy()

    d['hour']        = d['transaction_timestamp'].dt.hour
    d['dow']         = d['transaction_timestamp'].dt.dayofweek
    d['is_weekend']  = d['dow'].isin([5, 6]).astype(int)
    d['is_workhour'] = d['hour'].between(9, 18).astype(int)
    # BUGFIX: between(22, 6) always False because 22 > 6
    d['is_night']    = ((d['hour'] >= 22) | (d['hour'] <= 6)).astype(int)

    d['is_biz_mcc'] = d['mcc'].isin(BUSINESS_MCC).astype(int)
    d['is_food']    = d['mcc'].isin(['5411', '5812', '5814']).astype(int)
    d['is_large']   = (d['transaction_amount_kzt'] >= 50_000).astype(int)
    d['is_round']   = (d['transaction_amount_kzt'] % 1000 == 0).astype(int)
    d['is_intl']    = (d['country'] != 'Kazakhstan').astype(int)
    d['is_pos']     = (d['channel'] == 'POS').astype(int)

    agg = d.groupby('card_number').agg(
        txn_count        = ('transaction_amount_kzt', 'count'),
        total_amount     = ('transaction_amount_kzt', 'sum'),
        avg_amount       = ('transaction_amount_kzt', 'mean'),
        median_amount    = ('transaction_amount_kzt', 'median'),
        std_amount       = ('transaction_amount_kzt', 'std'),
        max_amount       = ('transaction_amount_kzt', 'max'),
        min_amount       = ('transaction_amount_kzt', 'min'),
        active_days      = ('transaction_date', 'nunique'),
        weekend_ratio    = ('is_weekend', 'mean'),
        workhour_ratio   = ('is_workhour', 'mean'),
        night_ratio      = ('is_night', 'mean'),      # исправлен
        unique_mcc       = ('mcc', 'nunique'),
        biz_mcc_ratio    = ('is_biz_mcc', 'mean'),
        food_ratio       = ('is_food', 'mean'),
        unique_merchants = ('merchant_id', 'nunique'),
        large_ratio      = ('is_large', 'mean'),
        round_ratio      = ('is_round', 'mean'),
        intl_ratio       = ('is_intl', 'mean'),
        unique_countries = ('country', 'nunique'),
        pos_ratio        = ('is_pos', 'mean'),
        recurring_ratio  = ('is_recurring', 'mean'),
        tokenized_ratio  = ('tokenized', 'mean'),
        label            = ('label', 'first'),
    ).reset_index()

    agg['txn_per_day'] = agg['txn_count'] / agg['active_days']
    agg['amount_cv']   = agg['std_amount'] / (agg['avg_amount'] + 1)
    agg['log_total']   = np.log1p(agg['total_amount'])
    agg['log_txn']     = np.log1p(agg['txn_count'])
    agg['std_amount']  = agg['std_amount'].fillna(0)

    return agg

print('Строим признаки...')
biz_feat  = build_features(biz)
cons_feat = build_features(cons)
print(f'Business: {len(biz_feat):,} карт x {len(biz_feat.columns)-2} признаков')
print(f'Consumer: {len(cons_feat):,} карт x {len(cons_feat.columns)-2} признаков')


## 5. Train / Validation / Test Split — без data leakage

**Ключевой момент:** делаем split по `card_number`, сохраняем индексы.
Все последующие переобучения используют **только train card_numbers** — никаких пересечений с val/test.


In [ ]:
all_cards = pd.concat([biz_feat, cons_feat], ignore_index=True).reset_index(drop=True)

FEATURES = [col for col in all_cards.columns
            if col not in ['card_number', 'label', 'round_ratio']]
# night_ratio теперь валидный признак после фикса бага (hour>=22)|(hour<=6)
# round_ratio оставляем исключённым — importance ≈ 0

X = all_cards[FEATURES].values
y = all_cards['label'].values

# Шаг 1: отделяем test (20%)
idx = np.arange(len(all_cards))
idx_temp, idx_test, y_temp, y_test = train_test_split(
    idx, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Шаг 2: из оставшихся 80% → train (75%) и val (25%)
idx_train, idx_val, y_train, y_val = train_test_split(
    idx_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp
)

# Матрицы для обучения
X_train, X_val, X_test = X[idx_train], X[idx_val], X[idx_test]

# Сохраняем card_numbers для каждого сплита — используем при переобучении
train_cards = set(all_cards.loc[idx_train, 'card_number'])
val_cards   = set(all_cards.loc[idx_val,   'card_number'])
test_cards  = set(all_cards.loc[idx_test,  'card_number'])

# Разбивка biz и cons карт по сплитам
biz_train_cards  = biz_feat[biz_feat['card_number'].isin(train_cards)]
cons_train_cards = cons_feat[cons_feat['card_number'].isin(train_cards)]
cons_val_cards   = cons_feat[cons_feat['card_number'].isin(val_cards)]
cons_test_cards  = cons_feat[cons_feat['card_number'].isin(test_cards)]

print(f'Train      : {len(idx_train):,} карт (60%)')
print(f'Validation : {len(idx_val):,} карт (20%)')
print(f'Test       : {len(idx_test):,} карт (20%)')
print(f'\nbiz в train  : {len(biz_train_cards):,}')
print(f'cons в train : {len(cons_train_cards):,}  <- среди них могут быть предприниматели')
print(f'Пересечений train/val: {len(train_cards & val_cards)}')   # должно быть 0
print(f'Пересечений train/test: {len(train_cards & test_cards)}') # должно быть 0


## 6. Базовые модели (до очистки)

In [ ]:
rf_base = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_base.fit(X_train, y_train)

xgb_base = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.1,
    random_state=RANDOM_STATE, n_jobs=-1, eval_metric='logloss'
)
xgb_base.fit(X_train, y_train)

auc_rf_base  = roc_auc_score(y_val, rf_base.predict_proba(X_val)[:, 1])
auc_xgb_base = roc_auc_score(y_val, xgb_base.predict_proba(X_val)[:, 1])

print(f'RF базовый   — Val AUC: {auc_rf_base:.4f}')
print(f'XGBoost базовый — Val AUC: {auc_xgb_base:.4f}')


## 7. Feature Importance

In [ ]:
fi = pd.DataFrame({
    'feature'   : FEATURES,
    'importance': rf_base.feature_importances_
}).sort_values('importance', ascending=False)

print('Важность признаков:')
print(fi.to_string(index=False))


## 8. Вариант 2: Итеративная очистка

**Фикс data leakage:** скорим только `cons_train_cards` — consumer карты из train.
Val и test карты не трогаем.


In [ ]:
print('=== Вариант 2: Итеративная очистка ===')

# Скорим только consumer карты которые были в train (не val/test!)
X_cons_train_only = cons_train_cards[FEATURES].values
cons_train_scored = cons_train_cards.copy()
cons_train_scored['score_base'] = rf_base.predict_proba(X_cons_train_only)[:, 1]

suspicious = (cons_train_scored['score_base'] >= 0.15).sum()
print(f'Consumer карт в train      : {len(cons_train_scored):,}')
print(f'Подозрительных (P >= 0.15) : {suspicious}')
print(f'Убираем их из train...')

# Надёжные потребители = те кто в train И у кого P < 0.15
reliable_cons = cons_train_scored[cons_train_scored['score_base'] < 0.15]

# Новый очищенный train (только карты из оригинального train сплита!)
clean_train = pd.concat([biz_train_cards, reliable_cons], ignore_index=True)
X_clean = clean_train[FEATURES].values
y_clean = clean_train['label'].values

print(f'Train до очистки  : {len(X_train):,} карт')
print(f'Train после очистки: {len(X_clean):,} карт')

rf_clean = RandomForestClassifier(
    n_estimators=200, class_weight='balanced',
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_clean.fit(X_clean, y_clean)

auc_clean = roc_auc_score(y_val, rf_clean.predict_proba(X_val)[:, 1])
print(f'\nRF базовый   Val AUC: {auc_rf_base:.4f}')
print(f'RF очищенный Val AUC: {auc_clean:.4f}')


## 9. Вариант 1: PU Learning — 2 итерации

**Consumer cards = Unlabeled**, не Negative. Это правильный фрейминг.

**Фикс circular bias:** делаем 2 итерации.
- Итерация 1: используем rf_base для поиска надёжных негативов
- Итерация 2: используем модель из итерации 1 — граница становится чище

**Фикс data leakage:** используем только `cons_train_cards` для обучения.


In [ ]:
print('=== Вариант 1: PU Learning (2 итерации) ===')

current_model = rf_base  # начинаем с базовой модели

for iteration in range(1, 3):  # 2 итерации
    print(f'\n--- Итерация {iteration} ---')

    # Скорим consumer карты из train (только train, не val/test!)
    X_cons_tr = cons_train_cards[FEATURES].values
    cons_tr_scored = cons_train_cards.copy()
    cons_tr_scored['pu_score'] = current_model.predict_proba(X_cons_tr)[:, 1]

    # Надёжные негативы = нижние 70% по скору среди train consumer
    threshold = cons_tr_scored['pu_score'].quantile(0.70)
    reliable_neg = cons_tr_scored[cons_tr_scored['pu_score'] <= threshold]
    uncertain    = cons_tr_scored[cons_tr_scored['pu_score'] >  threshold]

    print(f'Порог: P <= {threshold:.4f}')
    print(f'Надёжных потребителей : {len(reliable_neg):,}')
    print(f'Неопределённых        : {len(uncertain):,}')

    # Обучаем на biz_train_cards vs reliable_neg (только из train!)
    pu_train = pd.concat([biz_train_cards, reliable_neg], ignore_index=True)
    X_pu = pu_train[FEATURES].values
    y_pu = pu_train['label'].values

    rf_pu = RandomForestClassifier(
        n_estimators=200, class_weight='balanced',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    rf_pu.fit(X_pu, y_pu)

    auc_pu = roc_auc_score(y_val, rf_pu.predict_proba(X_val)[:, 1])
    print(f'Val AUC после итерации {iteration}: {auc_pu:.4f}')

    current_model = rf_pu  # следующая итерация использует улучшенную модель

print(f'\nRF базовый   Val AUC: {auc_rf_base:.4f}')
print(f'RF очищенный Val AUC: {auc_clean:.4f}')
print(f'RF PU (2 iter) AUC  : {auc_pu:.4f}')


## 10. XGBoost PU + Early Stopping

Вместо фиксированных 300 деревьев — XGBoost сам останавливается
когда val AUC перестаёт расти. Меньше переобучения, точнее граница.

Обучаем на тех же надёжных негативах что получили после 2 итераций PU.


In [ ]:
from xgboost import XGBClassifier

# Берём надёжные негативы из последней PU итерации
cons_tr_scored['pu_score'] = rf_pu.predict_proba(cons_train_cards[FEATURES].values)[:, 1]
threshold_final = cons_tr_scored['pu_score'].quantile(0.70)
reliable_final  = cons_tr_scored[cons_tr_scored['pu_score'] <= threshold_final]

pu_train_final = pd.concat([biz_train_cards, reliable_final], ignore_index=True)
X_pu_final = pu_train_final[FEATURES].values
y_pu_final = pu_train_final['label'].values

# Early stopping: разбиваем PU train на train/eval для XGBoost
X_xgb_tr, X_xgb_ev, y_xgb_tr, y_xgb_ev = train_test_split(
    X_pu_final, y_pu_final,
    test_size=0.2, random_state=RANDOM_STATE, stratify=y_pu_final
)

xgb_pu = XGBClassifier(
    n_estimators=1000,       # максимум — early stopping остановит раньше
    max_depth=6,
    learning_rate=0.05,      # медленнее учится = точнее
    subsample=0.8,           # случайная выборка строк = меньше переобучения
    colsample_bytree=0.8,    # случайная выборка признаков
    scale_pos_weight=(y_pu_final==0).sum() / (y_pu_final==1).sum(),
    random_state=RANDOM_STATE,
    n_jobs=-1,
    eval_metric='auc',
    early_stopping_rounds=30  # стоп если 30 раундов без улучшения
)

xgb_pu.fit(
    X_xgb_tr, y_xgb_tr,
    eval_set=[(X_xgb_ev, y_xgb_ev)],
    verbose=50  # печатать каждые 50 раундов
)

print(f'\nОстановился на дереве: {xgb_pu.best_iteration}')

auc_xgb_pu = roc_auc_score(y_val, xgb_pu.predict_proba(X_val)[:, 1])
print(f'\nRF PU (2 iter)         Val AUC: {auc_pu:.4f}')
print(f'XGBoost PU early stop  Val AUC: {auc_xgb_pu:.4f}')


## 11. Финальная оценка на Test Set

Сравниваем все модели на held-out test set.


In [ ]:
print('=== ФИНАЛЬНЫЙ ТЕСТ ===')

models = {
    'RF Базовый'           : rf_base,
    'XGBoost Базовый'      : xgb_base,
    'RF Очищенный'         : rf_clean,
    'RF PU (2 iter)'       : rf_pu,
    'XGBoost PU early stop': xgb_pu,
}

results = {}
for name, model in models.items():
    proba = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)
    auc   = roc_auc_score(y_test, proba)
    results[name] = {'auc': auc, 'proba': proba, 'preds': preds}
    print(f'{name:<22} Test AUC: {auc:.4f}')

best_name  = max(results, key=lambda k: results[k]['auc'])
best_model = models[best_name]
print(f'\nЛучшая модель: {best_name}')
print()
print(classification_report(
    y_test, results[best_name]['preds'],
    target_names=['Consumer', 'Business']
))


## 11. Финальный скоринг 80K consumer карт

Скорим **все** 80K consumer карт лучшей моделью.


In [ ]:
X_all_cons = cons_feat[FEATURES].values
cons_feat  = cons_feat.copy()
cons_feat['final_score'] = best_model.predict_proba(X_all_cons)[:, 1]

# Приоритизация
cons_feat['priority'] = 'Обычный потребитель'
cons_feat.loc[cons_feat['final_score'] >= 0.10, 'priority'] = 'Мониторинг'
cons_feat.loc[cons_feat['final_score'] >= 0.30, 'priority'] = 'Высокий риск'
cons_feat.loc[cons_feat['final_score'] >= 0.50, 'priority'] = 'Критический'

print('=== СРАВНЕНИЕ МОДЕЛЕЙ: сколько нашли ===')
for name, model in models.items():
    scores = model.predict_proba(X_all_cons)[:, 1]
    print(f'{name:<22} P>=0.10: {(scores>=0.10).sum():>5} | '
          f'P>=0.30: {(scores>=0.30).sum():>4} | '
          f'P>=0.50: {(scores>=0.50).sum():>4}')

print()
print('=== ИТОГ (PU Learning) ===')
print(cons_feat['priority'].value_counts().to_string())


## 12. Визуализация

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Feature Importance
fi2 = pd.DataFrame({
    'feature'   : FEATURES,
    'importance': rf_pu.feature_importances_
}).sort_values('importance', ascending=False).head(15)

axes[0, 0].barh(fi2['feature'][::-1], fi2['importance'][::-1], color='#C41E3A', alpha=0.85)
axes[0, 0].set_title('Feature Importance — RF PU Learning', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Importance')

# 2. ROC кривые
colors = ['#808080', '#FF8C00', '#2196F3', '#C41E3A', '#9B59B6']
for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['proba'])
    axes[0, 1].plot(fpr, tpr, lw=2, color=color, label=f'{name} ({res["auc"]:.4f})')
axes[0, 1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[0, 1].set_title('ROC-кривые — все модели', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('FPR'); axes[0, 1].set_ylabel('TPR')
axes[0, 1].legend(fontsize=8)

# 3. Распределение скоров
axes[1, 0].hist(cons_feat['final_score'], bins=60, color='#C41E3A', alpha=0.8)
axes[1, 0].axvline(0.10, color='orange', lw=2, linestyle='--', label='P=0.10 мониторинг')
axes[1, 0].axvline(0.30, color='red',    lw=2, linestyle='--', label='P=0.30 высокий риск')
axes[1, 0].axvline(0.50, color='black',  lw=2, linestyle='--', label='P=0.50 критический')
axes[1, 0].set_title('P(бизнес) — 80K consumer карт', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('P(бизнес)'); axes[1, 0].legend(fontsize=9)

# 4. Профиль предпринимателя
entrepreneurs = cons_feat[cons_feat['final_score'] >= 0.10]
ordinary      = cons_feat[cons_feat['final_score'] <  0.10]
metrics = ['biz_mcc_ratio', 'recurring_ratio', 'weekend_ratio', 'intl_ratio']
labels  = ['Бизнес MCC', 'Регулярные\nплатежи', 'Выходные', 'Международные']
x = np.arange(len(metrics))
axes[1, 1].bar(x - 0.2, [ordinary[m].mean()      for m in metrics],
               0.4, label='Обычный', color='#2ecc71', alpha=0.85)
axes[1, 1].bar(x + 0.2, [entrepreneurs[m].mean() for m in metrics],
               0.4, label='Предприниматель', color='#C41E3A', alpha=0.85)
axes[1, 1].set_xticks(x); axes[1, 1].set_xticklabels(labels, fontsize=10)
axes[1, 1].set_title('Профиль: предприниматель vs обычный', fontsize=13, fontweight='bold')
axes[1, 1].legend(fontsize=9)

plt.suptitle('Hidden Entrepreneurs Detection — Mastercard x AIESEC',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('results.png', dpi=150, bbox_inches='tight')
plt.show()


## 13. Сохранение результатов

In [ ]:
result = cons_feat[[
    'card_number', 'final_score', 'priority',
    'biz_mcc_ratio', 'recurring_ratio', 'pos_ratio',
    'total_amount', 'txn_count', 'unique_merchants'
]].sort_values('final_score', ascending=False)

result.to_csv('hidden_entrepreneurs.csv', index=False)

print(f'Сохранено: hidden_entrepreneurs.csv ({len(result):,} строк)')
print()
print('=== ИТОГОВЫЕ ВЫВОДЫ ===')
print(f'Лучшая модель    : {best_name}')
print(f'Test AUC         : {results[best_name]["auc"]:.4f}')
print(f'Data leakage     : устранён (split по card_number)')
print(f'is_night bug     : исправлен')
print()
print('Выявлено скрытых предпринимателей:')
print(f'  Критический (P>=0.5) : {(cons_feat["final_score"]>=0.50).sum()} карт')
print(f'  Высокий риск (P>=0.3): {((cons_feat["final_score"]>=0.30)&(cons_feat["final_score"]<0.50)).sum()} карт')
print(f'  Мониторинг (P>=0.1)  : {((cons_feat["final_score"]>=0.10)&(cons_feat["final_score"]<0.30)).sum()} карт')
print()
print('Бизнес-рекомендации:')
print('  Критический → предложить бизнес-карту + эквайринг')
print('  Высокий риск → targeted marketing через менеджера')
print('  Мониторинг  → наблюдение 2-3 месяца')
